In [5]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

model = "resnet"
ds_name = "emnist_letters"
split = "trainUval"

epoch = 48

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
dses = find_all_datasets("../../datasets/")
ds = dses[ds_name]

In [21]:
data_dir = "C:/home/eurovis_data/landscape_data_emnist_letters_last_best/"
strees_dir = "C:/home/eurovis_data/strees_emnist_letters_last_best/"

In [22]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[0]
for e in exps:
	if e.model == model and e.dataset.name == ds_name and e.split == split and e.epoch == epoch:
		exp = e
		break

assert exp.model == model
assert exp.dataset.name == ds_name
assert exp.split == split
assert exp.epoch == epoch

e

resnet (emnist_letters-trainUval), k=20, layer=1, epoch=48

In [23]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [24]:
import pyct as ct

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

homo_prop = 0.99
fns, counts = simpl.getHomoValleyPlot(order, wts, labels, homo_prop, partition) # type: ignore
fns_norm, counts_norm_min, counts_norm_max = simpl.getSimplificationPlot(order, wts)

In [25]:
import plotly.express as px

px.line(x=fns, y=counts, labels={"x": "Function Value", "y": "Number of Homogeneous Valleys"}, title=f"Homogeneous Valleys vs Function Value (Homogeneity Threshold = {homo_prop})")

In [26]:
px.line(x=fns_norm, y=counts_norm_min, labels={"x": "Function Value", "y": "Number of Valleys"}, title="Number of Valleys vs Function Value")

In [27]:
data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

fns_more, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = simpl.getHomoValleyPlotPlusCoverages(order, wts, labels, homo_prop, partition)

In [28]:
list(zip(maj_class_homo_cov, class_all_covs))

[([0.014642857142857143,
   0.0075,
   0.005535714285714285,
   0.004285714285714286,
   0.008035714285714285,
   0.0066071428571428574,
   0.011607142857142858,
   0.009642857142857142,
   0.009642857142857142,
   0.004642857142857143,
   0.0023214285714285715,
   0.017857142857142856,
   0.00625,
   0.005714285714285714,
   0.01017857142857143,
   0.005535714285714285,
   0.009821428571428571,
   0.004285714285714286,
   0.005535714285714285,
   0.0125,
   0.007857142857142858,
   0.011607142857142858,
   0.009642857142857142,
   0.004464285714285714,
   0.0066071428571428574,
   0.005714285714285714],
  [0.014642857142857143,
   0.0075,
   0.005535714285714285,
   0.004285714285714286,
   0.008035714285714285,
   0.0066071428571428574,
   0.011785714285714287,
   0.009642857142857142,
   0.009642857142857142,
   0.004642857142857143,
   0.0023214285714285715,
   0.017857142857142856,
   0.00625,
   0.005714285714285714,
   0.01017857142857143,
   0.005535714285714285,
   0.01,
   0.

In [39]:
idx = remaining_all.index(26)
idx, fns[idx], maj_class_homo_cov[idx], class_all_covs[idx], maj_class_homo_counts[idx]

(556,
 0.03161660581827164,
 [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.00035714285714285714,
  0.0,
  0.11267857142857143,
  0.6291071428571429,
  0.0,
  0.016428571428571428,
  0.92125,
  0.0,
  0.0,
  0.00017857142857142857,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.8960714285714285,
  0.8785714285714286,
  0.0,
  0.0],
 [0.6728571428571428,
  0.4757142857142857,
  0.7607142857142857,
  0.47714285714285715,
  0.7603571428571428,
  0.8235714285714286,
  0.4498214285714286,
  0.7514285714285714,
  0.11267857142857143,
  0.6291071428571429,
  0.8076785714285715,
  0.12696428571428572,
  0.92125,
  0.675,
  0.8171428571428572,
  0.6560714285714285,
  0.4132142857142857,
  0.7264285714285714,
  0.75875,
  0.7669642857142858,
  0.6282142857142857,
  0.7675,
  0.89625,
  0.8785714285714286,
  0.7908928571428572,
  0.7403571428571428],
 [0,
  0,
  0,
  0,
  0,
  0,
  2,
  0,
  6,
  1,
  0,
  10,
  1,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0])

In [70]:
classes_needed = 24 # 26 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.1 # at least 10% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.3 # at least 30% total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")

Uniform Interval: True
First IDX: 484 - 
484
582
0.00567586999386549
[(0.5333928571428571, 0.5333928571428571, 1), (0.40214285714285714, 0.40214285714285714, 2), (0.7498214285714285, 0.7498214285714285, 1), (0.32357142857142857, 0.32357142857142857, 2), (0.7441071428571429, 0.7441071428571429, 1), (0.7866071428571428, 0.7867857142857143, 1), (0.34089285714285716, 0.3410714285714286, 10), (0.7153571428571428, 0.7153571428571428, 1), (0.09303571428571429, 0.09303571428571429, 20), (0.5975, 0.5975, 4), (0.7775, 0.7775, 1), (0.11857142857142858, 0.11857142857142858, 34), (0.92125, 0.92125, 1), (0.6464285714285715, 0.6464285714285715, 1), (0.6880357142857143, 0.6880357142857143, 1), (0.5728571428571428, 0.5728571428571428, 2), (0.3169642857142857, 0.3171428571428571, 4), (0.6966071428571429, 0.6966071428571429, 2), (0.7064285714285714, 0.7064285714285714, 1), (0.69625, 0.69625, 1), (0.6025, 0.6025, 1), (0.7501785714285715, 0.7501785714285715, 1), (0.8960714285714285, 0.8960714285714285, 1),